[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilru1/dtw-to-attention/blob/main/notes/course_notes.ipynb)

# 1. Dynamic Time Warping (DTW)

## The Intuition

Unlike Euclidean distance, which matches points at the exact same timestamp (**xᵢ** to **yᵢ**), DTW looks for the "best match" or optimal alignment path between two series **x** and **y**. This allows it to handle temporal shifts and distortions.

## The Mathematical Optimization

DTW is defined as the minimum cost of all possible alignment paths in the set of admissible paths **A(x, x′)**:

**DTW_q(x, x′) = min_{π ∈ A(x, x′)} ∑_{(i,j) ∈ π} d(xᵢ, x′ⱼ)ᵖ**

An admissible path **π** must satisfy three constraints:

- **Boundary:** It must start at (1,1) and end at (n,m).  
- **Monotonicity:** The indices **i** and **j** can only increase.  
- **Continuity:** The path cannot skip timestamps (it moves in steps of 1).  

## The Algorithm

The problem is solved using **Dynamic Programming** with a complexity of **O(nm)**. The cumulative distance **γ_{i,j}** for any cell (i,j) is calculated as:

**γ_{i,j} = d(xᵢ, x′ⱼ)ᵖ + min(γ_{i-1,j}, γ_{i,j-1}, γ_{i-1,j-1})**

---

# 2. Refined DTW Constraints

To prevent "extreme" or unrealistic temporal shifts, **global constraints (masks)** are often applied to the search space:

- **Sakoe-Chiba Mask:** Restricts the path to a band of radius **r** around the diagonal.  
- **Itakura Parallelogram:** Constrains the path based on a maximum slope **s**.  

---

# 3. Soft-DTW: Making it Differentiable

## The Problem with Standard DTW

The **min** operator in standard DTW makes the function non-differentiable, which means it cannot be used directly as a loss function in Neural Networks or optimized easily with gradient descent.

## The Solution: Soft-Min

Soft-DTW replaces the "hard" minimum with a **soft-min operator**:

**soft-min_γ(a₁,...,aₙ) = -γ log ∑_i e^{-aᵢ / γ}**

- As **γ → 0**, it approaches the hard minimum.  
- The resulting Soft-DTW formula is a weighted sum of all possible paths, where "good" paths (low cost) receive high weights.

**soft-DTW(x, x′) = -γ log ∑_{π ∈ A(x, x′)} exp(-1/γ ∑_{(i,j) ∈ π} (xᵢ - x′ⱼ)²)**

---

# 4. Machine Learning Applications

The notebooks highlight several ways these metrics integrate into ML workflows:

- **Clustering:** DTW Barycenter Averaging (DBA) is used to find the average (centroid) of a set of time series under DTW. Because Soft-DTW is differentiable, its barycenters can be found via simple gradient descent.  
- **Kernel Methods:** The Global Alignment Kernel (GAK) allows DTW-like logic to be used in Support Vector Machines:

  **k_GAλ(x, x′) = exp(-γ · soft-DTW(x, x′))**

- **Neural Networks:** Soft-DTW can be used as a loss function for tasks requiring time-series alignment.

---

# 5. Toy Example: Computing DTW Distance

We compute the DTW distance between two time series:

**x = [1, 3, 5, 9]** and **x′ = [6, 4, 10]**  

We will use the squared Euclidean distance (**q = 2**) as the local cost:

**d(xᵢ, x′ⱼ) = (xᵢ - x′ⱼ)²**

## 1. Local Cost Matrix D

First, calculate the distance between every pair of points (**xᵢ, x′ⱼ**).  


## Local Cost Matrix D

| x ↓ / x′ → | 6       | 4       | 10      |
|------------|---------|---------|---------|
| 1          | (1−6)²=25 | (1−4)²=9  | (1−10)²=81 |
| 3          | (3−6)²=9  | (3−4)²=1  | (3−10)²=49 |
| 5          | (5−6)²=1  | (5−4)²=1  | (5−10)²=25 |
| 9          | (9−6)²=9  | (9−4)²=25 | (9−10)²=1  |


# 2. Accumulated Cost Matrix γ

Using the recurrence relation:  

**γ_{i,j} = d(xᵢ, x′ⱼ) + min(γ_{i-1,j}, γ_{i,j-1}, γ_{i-1,j-1})**,  

we build the cumulative cost matrix from the top-left corner.

### Step 1: Initialize
- **γ₁,₁ = 25**

### Step 2: Fill First Row/Column
- **γ₁,₂ = 9 + 25 = 34**  
- **γ₂,₁ = 9 + 25 = 34**

### Step 3: Fill the Rest
For each cell, take the **local cost** and add the **minimum of its neighbors** (top, left, top-left).

## Accumulated Cost Matrix γ

| x ↓ / x′ → | 6  | 4  | 10 |
|------------|----|----|----|
| 1          | 25 | 34 | 115 |
| 3          | 34 | 26 | 75  |
| 5          | 35 | 27 | 51  |
| 9          | 44 | 52 | 28  |

---

# 3. Final Results

- **DTW Value:** The value in the bottom-right cell **γ₄,₃ = 28** is the squared DTW distance.  

- **Optimal Path π\*** (backtracking from bottom-right by always choosing the smallest neighbor):  

  **(4,3) → (3,3) → (2,2) → (1,1)**  

  Corresponding matches:  

  **(9,10), (5,10), (3,4), (1,6)**  

- **Admissible Paths:** In a 4×3 matrix, there are multiple paths that satisfy the step rules; in this toy example, 3 main paths exist from start to finish.

---

# 4. Comparison: Soft-DTW

- **Soft-min:** Instead of taking the hard minimum, Soft-DTW computes a **weighted sum over all paths** using the log-sum-exp formula.  
- **Smoothing Parameter γ:**  
  - Higher γ → more smoothing (all paths contribute more evenly)  
  - γ → 0 → recovers the original DTW value (hard minimum)  
- **Benefit:** Makes DTW **differentiable**, allowing it to be used as a **loss function** in neural networks.


Building on the previous discussion of metrics, these slides shift focus toward **time series embedding**—the process of transforming temporal data into static, k-dimensional vectors so that classical machine learning techniques can be applied.

A central theme in these embeddings is the use of **shapelets**, which are representative subsequences that characterize specific classes within a dataset.

---

## 1. Shapelets: The Building Blocks

A shapelet is defined by its ability to discriminate between classes based on distance:

* 
**Good Shapelet**: Typically has a **low distance** to time series within its own class and a **high distance** to series in other classes.


* 
**Discriminative Power**: This is often measured using criteria like the **Gini index** or **mutual information gain** to determine how well a candidate shapelet separates data points.



---

## 2. Evolution of Shapelet Methods

The theory has evolved from simple decision trees to complex neural network integrations:

| Method | Approach | Pros | Cons |
| --- | --- | --- | --- |
| <br>**Shapelet Decision Tree** 

 | Uses an exhaustive set of candidates to build a tree.

 | Highly interpretable.

 | Lower accuracy; requires exhaustive search.

 |
| <br>**Shapelet Transform** 

 | Transforms series into a vector of distances to the  best shapelets.

 | Improved accuracy.

 | Computationally expensive; loses interpretability.

 |
| <br>**Learning Shapelets** 

 | Shapelets are learned as parameters of a model via cross-entropy loss.

 | Maintains accuracy without exhaustive search.

 | Interpretability is decreased.

 |

---

## 3. Advanced Neural Architectures

The slides conclude by connecting shapelets to modern deep learning:

* 
**Localized Random Shapelets**: Uses random shapelets combined with a multilayer perceptron and regularization techniques like **semi-sparse group lasso** to select the most relevant features.


* 
**Adversarial Regularization**: A framework that uses a **classifier** and a **discriminator** to refine shapelets, essentially treating convolution kernels as learned shapelets.


* 
**Convolutional Neural Networks (CNNs)**: Time series are processed through layers of **convolutions** (e.g., size 5) and **max-pooling** (e.g., size 2 or 4) to automatically extract feature maps.



Would you like me to explain the difference between the **Lasso** and **Group Lasso** regularization mentioned for localized shapelets?

In the context of localized random shapelets, regularization is used to select the most discriminative features (distances to shapelets and their positions) while keeping the model sparse. The slides mention three types of regularization used in the Multi-Layer Perceptron (MLP) framework:

### 1. Lasso (Least Absolute Shrinkage and Selection Operator)

* 
**Purpose**: Encourages sparsity at the individual feature level.


* **Mechanism**: It adds a penalty proportional to the absolute value of the weights ( norm).
* **Effect**: It can force some feature weights to be exactly zero, effectively "deleting" irrelevant shapelets from the model.

### 2. Group Lasso

* 
**Purpose**: Encourages sparsity at the group level.


* **Mechanism**: Instead of penalizing individual weights, it penalizes groups of related variables together.
* 
**In this context**: Since each shapelet provides both a distance  and a position , Group Lasso ensures that if a shapelet is deemed irrelevant, both its distance and its position are removed from the model simultaneously.



### 3. Semi-Sparse Group Lasso

* 
**Mechanism**: A hybrid approach that combines the strengths of both Lasso and Group Lasso.


* 
**Effect**: It can discard entire groups (an entire shapelet) or discard individual features within a group.


* **Benefit**: This is highly effective for localized shapelets because it allows the model to decide if a shapelet is useful only because of its **shape** (distance), only because of **where it appears** (position), or both.

---

## Summary of Embedding Techniques

To wrap up the theory from these notebooks, here is how the representation of time series has evolved:

| Technique | Core Logic | Key Components |
| --- | --- | --- |
| **Bag-of-Words** | Discretization and frequency | SAX, SFA, 1D-SIFT 

 |
| **Shapelet-based** | Local subsequence matching | Decision trees, Transforms, or Learned kernels 

 |
| **Random Kernels** | Rapid, random feature extraction | ROCKET 

 |
| **CNNs** | Hierarchical feature learning | Convolutions, Max-Pooling, and Feature Maps 

 |

Would you like me to create a summary table comparing **Dynamic Time Warping** (from the first notebook) against **Shapelet-based embeddings** (from the second) to help you choose between them for a specific project?

Absolutely! Let’s clarify the distinction clearly and link it to your **localized shapelets** context:

---

## Lasso vs. Group Lasso Regularization

### 1. **Lasso (L1 Regularization)**

* **Formula:**

[
\text{Loss} + \lambda \sum_{i} |w_i|
]

* **Effect:**

  * Encourages **sparsity at the individual feature level**.
  * Some weights are exactly zero → automatic **feature selection**.
* **In the shapelets context:**

  * Helps select **individual shapelets** that are most discriminative.
  * Each distance feature to a shapelet can be zeroed out independently.

---

### 2. **Group Lasso**

* **Formula:**

[
\text{Loss} + \lambda \sum_{g \in \text{groups}} | \mathbf{w}_g |_2
]

* **Effect:**

  * Encourages **sparsity at the group level**, not individual features.
  * Either all features in a group are kept, or all are zeroed.
* **In the shapelets context:**

  * Shapelets can be grouped (e.g., all random shapelets of the same length or for the same temporal region).
  * The regularization keeps **entire groups of related shapelets** or removes them entirely, rather than picking individual ones.

---

### 🔑 Key Difference

| Feature           | Lasso                               | Group Lasso                                                    |
| ----------------- | ----------------------------------- | -------------------------------------------------------------- |
| Sparsity level    | Individual features                 | Groups of features                                             |
| Feature selection | Per feature                         | Per group                                                      |
| Shapelet example  | Picks a few shapelets independently | Picks sets of shapelets together (e.g., same length or region) |

---

💡 **Intuition:**

Think of **Lasso** as picking a few “winning players” from a team, while **Group Lasso** either keeps the whole “team” or benches it entirely.

---

If you like, I can also **draw a mini diagram showing shapelets grouped by temporal region and which ones Lasso vs. Group Lasso would select**—it’s really intuitive for slides.

Do you want me to do that?
